<a href="https://colab.research.google.com/github/areebaeman234-ux/ML-Internship/blob/main/Copy_of_w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions


### Finding 1: Random Forest feature importance for Health Score

The paper reports that Average Position had the highest Random Forest feature importance at 43%, followed by Impressions at 32% and Scroll Depth at 15%.

**My methodology question:**

How is the Health Score label constructed, and are the features used by the Random Forest also components of that label?

The paper states that Health Score is constructed from Impressions, Position, CTR, and Scroll Depth. Therefore, these features overlap directly with the target being predicted.

I would not treat the 43% importance for Average Position as evidence that Average Position independently causes Health Score. Instead, I would interpret it as descriptive model behavior within a target that already contains related inputs.

This is a constructive methodology question because understanding the target construction helps determine what the feature-importance result actually tells us.

---

### Finding 2: Logistic Regression for Growth

The paper reports 71% holdout accuracy for a Logistic Regression model that separates growing pages from declining pages.

**My methodology question:**

Does the 80/20 holdout validation design represent the real prediction setting well enough to support this result?

Growth is defined using changes in impressions over time. Because the outcome is time-based, I would want to check whether a random holdout could place related or temporally connected observations on both sides of the split.

A time-aware or grouped validation design could provide a more conservative test of whether the measured performance transfers to unseen future observations or unseen groups.

I am not saying that the reported 71% accuracy is incorrect. My question is whether the validation design supports how broadly that accuracy should be interpreted.

## 2. My model under an honest split (before/after)

## Before vs After: Validation Design

The Week-5 model used a random 80/20 train-test split. Under this design, the
Random Forest had an observed R² of -0.4309 and an MAE of 0.000958.

For Week 6, I changed the validation design to a client-grouped 80/20 split
using the anonymized `client_hash_id`. The training and testing sets contained
44 and 11 clients respectively, with zero client overlap.

Under the grouped split, the Random Forest had an observed R² of -0.0370 and
an MAE of 0.001392.

The measured performance changed when the validation design changed. The
grouped result is more appropriate for assessing performance on clients that
were not present in training.

The Random Forest remained slightly worse than the grouped mean baseline on
R² (-0.0370 versus -0.0001). Therefore, this experiment does not provide
evidence that the current model improves on the simple baseline for unseen
clients.                                                                       What I learned

The random split and client-grouped split produced different measured results.

This shows why validation design matters for this task. A random split can
place observations from the same client in both training and testing data.

The client-grouped split prevents this overlap and gives a more conservative
test of generalization to unseen clients.

The current Random Forest should therefore be treated as a decision-support
experiment rather than a production-ready predictor.

In [ ]:
# ============================================
# WEEK 6 — LOAD DATA
# ============================================

!pip install -q duckdb pyarrow scikit-learn

import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

hf_token = userdata.get("hf_token")

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')"
)

query = """
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
LIMIT 10000
"""

df = con.execute(query).df()

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())
# ============================================
# WEEK 6 — LOAD MULTI-CLIENT MARCH DATA
# ============================================

query = """
WITH sampled AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY client_hash_id
            ORDER BY content_hash_id
        ) AS client_row
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
)

SELECT * EXCLUDE (client_row)
FROM sampled
WHERE client_row <= 180
"""

df = con.execute(query).df()

print("Rows:", len(df))
print("Unique clients:", df["client_hash_id"].nunique())
# ============================================
# CREATE CTR TARGET
# ============================================

df["gsc_ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"].replace(0, np.nan)
)

df["gsc_ctr"] = df["gsc_ctr"].fillna(0)

print("CTR created.")
print(df["gsc_ctr"].describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 10000
Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 9900
Unique clients: 55
CTR created.
count    9900.000000
mean        0.000640
std         0.012483
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: gsc_ctr, dtype: float64


In [ ]:
# ============================================
# FEATURES, TARGET AND GROUP
# ============================================

features = [
    "gsc_avg_position",
    "gsc_impressions"
]

X = df[features].copy()
y = df["gsc_ctr"].copy()
groups = df["client_hash_id"].copy()

X = X.fillna(0)

print("Features:", features)
print("Target: gsc_ctr")
print("Grouping variable: client_hash_id")
print("Rows:", len(X))
print("Unique clients:", groups.nunique())

Features: ['gsc_avg_position', 'gsc_impressions']
Target: gsc_ctr
Grouping variable: client_hash_id
Rows: 9900
Unique clients: 55


In [ ]:
# ============================================
# BEFORE: WEEK-5 STYLE RANDOM SPLIT
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

random_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

random_model.fit(
    X_train_random,
    y_train_random
)

random_pred = random_model.predict(X_test_random)

random_r2 = r2_score(
    y_test_random,
    random_pred
)

random_mae = mean_absolute_error(
    y_test_random,
    random_pred
)

print("Week-5 style random split")
print("R²:", round(random_r2, 4))
print("MAE:", round(random_mae, 6))

Week-5 style random split
R²: -0.4309
MAE: 0.000958


In [ ]:
# ============================================
# AFTER: WEEK-6 CLIENT-GROUPED SPLIT
# ============================================

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train_group))
print("Testing rows:", len(X_test_group))

print("Training clients:", groups_train.nunique())
print("Testing clients:", groups_test.nunique())

overlap = set(groups_train) & set(groups_test)

print("Overlapping clients:", len(overlap))

Training rows: 7920
Testing rows: 1980
Training clients: 44
Testing clients: 11
Overlapping clients: 0


In [ ]:
# ============================================
# RANDOM FOREST — CLIENT-GROUPED VALIDATION
# ============================================

group_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

group_model.fit(
    X_train_group,
    y_train_group
)

group_pred = group_model.predict(X_test_group)

group_r2 = r2_score(
    y_test_group,
    group_pred
)

group_mae = mean_absolute_error(
    y_test_group,
    group_pred
)

print("Week-6 client-grouped split")
print("R²:", round(group_r2, 4))
print("MAE:", round(group_mae, 6))

Week-6 client-grouped split
R²: -0.037
MAE: 0.001392


In [ ]:
# ============================================
# CLIENT-GROUPED BASELINE
# ============================================

baseline_value = y_train_group.mean()

baseline_pred = np.full(
    len(y_test_group),
    baseline_value
)

baseline_r2 = r2_score(
    y_test_group,
    baseline_pred
)

baseline_mae = mean_absolute_error(
    y_test_group,
    baseline_pred
)

print("Client-grouped baseline")
print("R²:", round(baseline_r2, 4))
print("MAE:", round(baseline_mae, 6))
# ============================================
# BEFORE vs AFTER
# ============================================

comparison = pd.DataFrame({
    "Validation Design": [
        "Week-5 Random 80/20",
        "Week-6 Client-Grouped 80/20"
    ],
    "R2": [
        random_r2,
        group_r2
    ],
    "MAE": [
        random_mae,
        group_mae
    ]
})

comparison

Client-grouped baseline
R²: -0.0001
MAE: 0.00141


,Validation Design,R2,MAE
0,Week-5 Random 80/20,-0.430883,0.000958
1,Week-6 Client-Grouped 80/20,-0.037042,0.001392


In [ ]:
# ============================================
# HONEST TEST RESULTS
# ============================================

honest_results = pd.DataFrame({
    "Method": [
        "Mean Baseline",
        "Random Forest"
    ],
    "R2": [
        baseline_r2,
        group_r2
    ],
    "MAE": [
        baseline_mae,
        group_mae
    ]
})

honest_results

,Method,R2,MAE
0,Mean Baseline,-0.000109,0.001410
1,Random Forest,-0.037042,0.001392


## 3. Leakage audit

### Target Definition:
**CTR = clicks / impressions**

### Key Findings:

| Feature | Leakage Risk | Why |
|---------|--------------|-----|
| **gsc_impressions** |  HIGH | Used in CTR denominator (target calculation) |
| **gsc_clicks** |  HIGH | Used in CTR numerator (not used as feature) |
| **gsc_avg_position** |  CHECK TIMING | Same period as target; missing when impressions = 0 |
| **client_hash_id** |  SAFE | Used only for grouping, not prediction |

### Zero-Impression Issue:

- **7,432 rows** (74% of sample) have zero impressions
- CTR = 0 for these rows (after division fix)
- Model performance reflects both zero-activity records and measured CTR

### Correlation Review:

| Pair | Correlation |
|------|-------------|
| CTR vs Impressions | ~0.034 |
| CTR vs Position | ~-0.043 |

*These are descriptive only - not causal.*

### Missingness:

- `gsc_avg_position` has 7,432 missing values
- This matches zero-impression rows exactly
- Position is unavailable when no impressions exist

### Audit Decision:

| Feature | Decision |
|---------|----------|
| gsc_impressions |  Use with qualification - part of target calculation |
| gsc_clicks |  Exclude - directly in CTR numerator |
| gsc_avg_position |  Flag for timing review |
| client_hash_id |  Safe - grouping only |

### Bottom Line:

> "The model should be treated as a development experiment, not as evidence of causal or production-ready prediction. The feature set overlaps with the target definition, which limits interpretation."

In [ ]:
# ============================================
# SECTION 3 — LEAKAGE AUDIT
# TARGET CONSTRUCTION CHECK
# ============================================

print("Target definition:")
print("gsc_ctr = gsc_clicks / gsc_impressions")

print("\nTarget statistics:")
print(df["gsc_ctr"].describe())

print("\nZero-impression rows:")
print((df["gsc_impressions"] == 0).sum())

print("\nRows with clicks:")
print((df["gsc_clicks"] > 0).sum())

Target definition:
gsc_ctr = gsc_clicks / gsc_impressions

Target statistics:
count    9900.000000
mean        0.000640
std         0.012483
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: gsc_ctr, dtype: float64

Zero-impression rows:
7432

Rows with clicks:
246


In [ ]:
# ============================================
# FEATURE-LEVEL LEAKAGE AUDIT
# ============================================

leakage_audit = pd.DataFrame({
    "Feature": [
        "gsc_avg_position",
        "gsc_impressions",
        "gsc_clicks",
        "client_hash_id"
    ],

    "Used_in_Model": [
        "Yes",
        "Yes",
        "No",
        "No"
    ],

    "Used_in_CTR_Calculation": [
        "No",
        "Yes - denominator",
        "Yes - numerator",
        "No"
    ],

    "Leakage_or_Timing_Risk": [
        "Timing check",
        "High",
        "High",
        "Low"
    ],

    "Decision": [
        "Review availability",
        "Flag for prediction-time review",
        "Do not use",
        "Use only for grouping"
    ]
})

leakage_audit

,Feature,Used_in_Model,Used_in_CTR_Calculation,Leakage_or_Timing_Risk,Decision
0,gsc_avg_position,Yes,No,Timing check,Review availability
1,gsc_impressions,Yes,Yes - denominator,High,Flag for prediction-time review
2,gsc_clicks,No,Yes - numerator,High,Do not use
3,client_hash_id,No,No,Low,Use only for grouping


In [ ]:
# ============================================
# VERIFY CTR CALCULATION
# ============================================

check_ctr = df[
    df["gsc_impressions"] > 0
].copy()

check_ctr["calculated_ctr"] = (
    check_ctr["gsc_clicks"] /
    check_ctr["gsc_impressions"]
)

max_difference = (
    check_ctr["calculated_ctr"] -
    check_ctr["gsc_ctr"]
).abs().max()

print(
    "Maximum difference between calculated CTR "
    "and target CTR:",
    max_difference
)

Maximum difference between calculated CTR and target CTR: 0.0


In [ ]:
# ============================================
# CORRELATION AUDIT
# ============================================

correlation_data = df[
    [
        "gsc_ctr",
        "gsc_clicks",
        "gsc_impressions",
        "gsc_avg_position"
    ]
].corr()

correlation_data

,gsc_ctr,gsc_clicks,gsc_impressions,gsc_avg_position
gsc_ctr,1.000000,0.211117,0.033595,-0.043043
gsc_clicks,0.211117,1.000000,0.632885,-0.072857
gsc_impressions,0.033595,0.632885,1.000000,-0.087246
gsc_avg_position,-0.043043,-0.072857,-0.087246,1.000000


In [ ]:
# ============================================
# TIMING / PREDICTION-AVAILABILITY CHECK
# ============================================

timing_check = df[
    [
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "gsc_ctr"
    ]
].head(10)

timing_check

,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_ctr
0,2026-03-12,0,0,NaN,0.0
1,2026-03-11,0,0,NaN,0.0
2,2026-03-15,0,0,NaN,0.0
3,2026-03-07,0,0,NaN,0.0
4,2026-03-17,0,0,NaN,0.0
5,2026-03-20,0,0,NaN,0.0
6,2026-03-14,0,0,NaN,0.0
7,2026-03-19,0,0,NaN,0.0
8,2026-03-27,0,0,NaN,0.0
9,2026-03-28,0,0,NaN,0.0


In [ ]:
# ============================================
# FEATURE VARIATION CHECK
# ============================================

feature_variation = pd.DataFrame({
    "Feature": [
        "gsc_avg_position",
        "gsc_impressions"
    ],
    "Unique_values": [
        df["gsc_avg_position"].nunique(),
        df["gsc_impressions"].nunique()
    ],
    "Missing_values": [
        df["gsc_avg_position"].isna().sum(),
        df["gsc_impressions"].isna().sum()
    ]
})

feature_variation

,Feature,Unique_values,Missing_values
0,gsc_avg_position,1589,7432
1,gsc_impressions,339,0


In [ ]:
# ============================================
# SAVE LEAKAGE AUDIT
# ============================================

import os

os.makedirs("work/outputs", exist_ok=True)

leakage_audit.to_csv(
    "work/outputs/w06_leakage_audit.csv",
    index=False
)

correlation_data.to_csv(
    "work/outputs/w06_feature_correlations.csv"
)

feature_variation.to_csv(
    "work/outputs/w06_feature_variation.csv",
    index=False
)

print("Leakage audit outputs saved.")

Leakage audit outputs saved.


## 4. Claim rewrite

In the development experiment, gsc_avg_position showed model importance, but the result is descriptive and does not establish that position causes changes in CTR."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.